In [2]:
# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table, callback_context
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Uses Feero_animal_shelter_crud_authentication python file to import AnimalShelter class
from Feero_animal_shelter_crud_authentication import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = os.getenv('MONGO_USERNAME')
password = os.getenv('MONGO_PASSWORD')
HOST = 'localhost'
PORT = 27017
DB = 'aac'
COL = 'animals'

# Connect to database via CRUD Module
shelter = AnimalShelter(username, password)

# Class read method must support return of list object and accept projection json input
# Sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# Method to drop the id column without crashing the database
df.drop(columns=['_id'],inplace=True)

#########################
# Dashboard Layout / View
#########################

app = Dash(__name__)

# Adds the company's logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Develops the layout of the dashboard
app.layout = html.Div([
    # Login container
    html.Div(id="login-container", children=[
        html.Center(html.H2("Please enter your credentials:")),
        html.Center(dcc.Input(id="username-input", type="text", placeholder="Username")),
        html.Center(dcc.Input(id="password-input", type="password", placeholder="Password")),
        html.Center(html.Button("Login", id="login-button")),
        html.Center(html.Div(id="login-message", style={"color":"red", "marginTop":"10px"}))
    ]),
    
    # Dashboard container
    html.Div(id='dashboard-container', style={'display':'none'}, children=[
    # Center image
    html.Center(
        # Anchor tag (wrap) image
        html.A(
            # Display image in dashboard
            html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode())),
            # Provide desired link for wrapped image (template uses SNHU link, but would be changed to company site when deployed)
            href='https://www.snhu.edu',
            # Open desired link in new tab
            target='_blank'
        )
    ),
    html.Center(html.B(html.H1('Animal Shelter Dashboard'))),
    html.Center(html.B(html.H2("Nicholas Feero | CS-499 Enhancement #3 Capstone"))),
    html.Hr(),
    html.Div(className='buttonRow',
            style={'display' : 'flex'},
                children=[
                    html.Button(id='submit-button-one', n_clicks=0, children='No Filter'),
                    html.Button(id='submit-button-two', n_clicks=0, children='Water Rescue'),
                    html.Button(id='submit-button-three', n_clicks=0, children='Mountain or Wilderness Rescue'),
                    html.Button(id='submit-button-four', n_clicks=0, children='Disaster Rescue or Individual Tracking'),
                ]),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        # Create interactive data table
        editable=False,
        filter_action="native",
        sort_action="native",
        column_selectable="single",
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10,

    ),

    html.Br(),
    html.Hr(),

# This sets up the dashboard so that the pie chart and geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display': 'flex', 'height': '500px'},  # Set row height
        children=[
            html.Div(id='graph-id', style={'flex': '1'}),
            html.Div(
                id='map-id',
                style={'flex': '1', 'height': '100%'}  # Fill the container
            )
        ])
    ])
])

#################
# Login callback
#################

@app.callback(
    Output("login-container", "style"),
    Output("dashboard-container", "style"),
    Output("login-message", "children"),
    Input("login-button", "n_clicks"),
    State("username-input", "value"),
    State("password-input", "value"),
    prevent_initial_call=True
)
def login(n_clicks, username_check, password_check):
    if username_check == username and password_check == password:
        return {"display":"none"}, {"display":"block"}, ""
    else:
        return {"display":"block"}, {"display":"none"}, "Invalid username or password."

#############################################
# Interaction Between Components / Controller
#############################################

# Callback to allow button filter switching
@app.callback(
    Output('datatable-id', "data"),
    [Input('submit-button-one', 'n_clicks'),
     Input('submit-button-two', 'n_clicks'),
     Input('submit-button-three', 'n_clicks'),
     Input('submit-button-four', 'n_clicks')
    ])

# Define clicking function with parameter of 5 inputs
def on_click(button1, button2, button3, button4):
    # Set variable to callback_context function import
    ctx = callback_context
    # Define a button's actions with the context of callback
    # Create if and elif statements for each button press, and provide the class read method as a default
    button_id = ctx.triggered[0]['prop_id'].split('.')[0]
    if button_id == 'submit-button-one':
        df = pd.DataFrame.from_records(shelter.read({}))
    elif button_id == 'submit-button-two':
        df = pd.DataFrame.from_records(shelter.read({"animal_type" : "Dog", "breed" : {"$in" : ["Labrador Retriever Mix", 
                                                    "Chesapeake Bay Retriever", "Newfoundland"]},
                                                    "sex_upon_outcome" : "Intact Female", 
                                                    "age_upon_outcome_in_weeks" : {"$gte":26}, 
                                                    "age_upon_outcome_in_weeks" : {"$lte":156}}))
    elif button_id == 'submit-button-three':
        df = pd.DataFrame.from_records(shelter.read({"animal_type" : "Dog","breed" 
                                                        : {"$in" : ["German Shepherd", "Alaskan Malamute", 
                                                        "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
                                                        "sex_upon_outcome" : "Intact Male",
                                                        "age_upon_outcome_in_weeks" : {"$gte":26},
                                                        "age_upon_outcome_in_weeks" : {"$lte":156}}))
    elif button_id == 'submit-button-four':
        df = pd.DataFrame.from_records(shelter.read({"animal_type" : "Dog","breed" 
                                                        : {"$in" : ["Doberman Pinscher", "German Shepherd", 
                                                        "Golden Retriever", "Bloodhound", "Rottweiler"]},
                                                        "sex_upon_outcome" : "Intact Male",
                                                        "age_upon_outcome_in_weeks" : {"$gte":20},
                                                        "age_upon_outcome_in_weeks" : {"$lte":300}}))
    else:
        df = pd.DataFrame.from_records(shelter.read({})) 
    
    # Cleanup Mongo _id field
    df.drop(columns=['_id'],inplace=True)
    return df.to_dict('records')

def update_dashboard(filter_type):
# Filter interactive data table with MongoDB queries
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    data=df.to_dict('records')
      
    return (data,columns)

# Display the breeds of animal based on quantity represented in the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # Add code for pie chart display
    if viewData is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    
     # Count breeds
    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']

    # Combine small categories into 'Other' to prevent visual clutter on non-filtered display
    # Establish breed threshold of 10
    threshold = 10
    # If there are more than 10 breeds in the selected data, only display the top 10 most common breeds
    if len(breed_counts) > threshold:
        # Set 'top_breeds' as top 10 breeds
        top_breeds = breed_counts[:threshold]
        # Acquire total count of less common breeds
        other_count = breed_counts[threshold:]['count'].sum()
        # Rstablish breed 'Other' with count of 'other_count'
        other_row = pd.DataFrame([{'breed': 'Other', 'count': other_count}])
        # Group most common and least common breeds to be read within the same context/dataset
        breed_counts = pd.concat([top_breeds, other_row], ignore_index=True)

    # Create pie chart object
    chart = px.pie(
        breed_counts,
        names='breed',
        values='count',
        title='Breed Distribution',
        color_discrete_sequence=px.colors.qualitative.Set3,
    )
    
    # Updatable layout for chart
    chart.update_layout(
        showlegend=True,
        legend_title_text='Breed'
    )

    return [dcc.Graph(figure=chart)]
    
# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form ofa list. 
# For this application, single row selection is permitted, so there is only one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Since only single row selection is allowed, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]

     # Get lat/lon (make sure these are correct for your dataset)
    lat = float(dff.iloc[row, 12])
    lon = float(dff.iloc[row, 13])
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '100%', 'height': '100%'}, center=[lat, lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,12],dff.iloc[row,13]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]

app.run(debug=True, port=8050)